# Olist E-commerce: Loading and Exploring the Data

**Goal:** load the 9 Olist tables into a DuckDB database, understand how they connect,
and check the data quality before any analysis.

In [1]:
import duckdb

In [2]:
%load_ext sql
con = duckdb.connect("../data/olist.duckdb")
%sql con
%config SqlMagic.displaylimit = 50

In [3]:
%%sql
CREATE OR REPLACE TABLE customers AS SELECT * FROM read_csv_auto('../data/olist_customers_dataset.csv');
CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('../data/olist_orders_dataset.csv');
CREATE OR REPLACE TABLE order_items AS SELECT * FROM read_csv_auto('../data/olist_order_items_dataset.csv');
CREATE OR REPLACE TABLE payments AS SELECT * FROM read_csv_auto('../data/olist_order_payments_dataset.csv');
CREATE OR REPLACE TABLE reviews AS SELECT * FROM read_csv_auto('../data/olist_order_reviews_dataset.csv');
CREATE OR REPLACE TABLE products AS SELECT * FROM read_csv_auto('../data/olist_products_dataset.csv');
CREATE OR REPLACE TABLE sellers AS SELECT * FROM read_csv_auto('../data/olist_sellers_dataset.csv');
CREATE OR REPLACE TABLE category_translation AS SELECT * FROM read_csv_auto('../data/product_category_name_translation.csv');

Running query in 'DuckDBPyConnection'

99441 rows affected.

99441 rows affected.

112650 rows affected.

103886 rows affected.

99224 rows affected.

32951 rows affected.

3095 rows affected.

71 rows affected.

Count
71


In [4]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'payments', COUNT(*) FROM payments
UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'sellers', COUNT(*) FROM sellers
UNION ALL SELECT 'category_translation', COUNT(*) FROM category_translation
ORDER BY row_count DESC;

Running query in 'DuckDBPyConnection'

table_name,row_count
order_items,112650
payments,103886
customers,99441
orders,99441
reviews,99224
products,32951
sellers,3095
category_translation,71


In [5]:
%%sql
DESCRIBE orders;

Running query in 'DuckDBPyConnection'

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
order_status,VARCHAR,YES,None,None,None
order_purchase_timestamp,TIMESTAMP,YES,None,None,None
order_approved_at,TIMESTAMP,YES,None,None,None
order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
order_delivered_customer_date,TIMESTAMP,YES,None,None,None
order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [6]:
%%sql
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT order_id) AS unique_orders
FROM orders;

Running query in 'DuckDBPyConnection'

total_rows,unique_orders
99441,99441


In [7]:
%%sql
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
       COUNT(DISTINCT customer_unique_id) AS real_customers
FROM customers;

Running query in 'DuckDBPyConnection'

customer_ids,real_customers
99441,96096


In [8]:
%%sql
SELECT COUNT(*) AS item_rows,
       COUNT(DISTINCT order_id) AS orders_with_items
FROM order_items;

Running query in 'DuckDBPyConnection'

item_rows,orders_with_items
112650,98666


In [9]:
%%sql
SELECT order_status,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY orders DESC;

Running query in 'DuckDBPyConnection'

order_status,orders,pct
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.3
created,5,0.01
approved,2,0.0


In [10]:
%%sql
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM orders;

Running query in 'DuckDBPyConnection'

first_order,last_order
2016-09-04 21:15:19,2018-10-17 17:30:18


In [11]:
%%sql
SELECT date_trunc('month', order_purchase_timestamp) AS month,
       COUNT(*) AS orders
FROM orders
GROUP BY month
ORDER BY month;

Running query in 'DuckDBPyConnection'

month,orders
2016-09-01 00:00:00,4
2016-10-01 00:00:00,324
2016-12-01 00:00:00,1
2017-01-01 00:00:00,800
2017-02-01 00:00:00,1780
2017-03-01 00:00:00,2682
2017-04-01 00:00:00,2404
2017-05-01 00:00:00,3700
2017-06-01 00:00:00,3245
2017-07-01 00:00:00,4026


In [12]:
%%sql
SELECT COUNT(*) - COUNT(order_approved_at) AS missing_approved,
       COUNT(*) - COUNT(order_delivered_customer_date) AS missing_delivered
FROM orders;

Running query in 'DuckDBPyConnection'

missing_approved,missing_delivered
160,2965


## What I learned about the data

- **8 tables loaded** into DuckDB (geolocation skipped: not needed for our questions).
  `orders` is the central table; everything else joins to it.
- **Keys:**
  - `order_id` uniquely identifies an order (99,441 orders).
  - `customer_id` is created **per order**. The real person is `customer_unique_id`
    (96,096 people). Repeat-customer analysis must use `customer_unique_id`.
  - `order_items` has one row **per item** (112,650 items in 98,666 orders), so joins
    to it can multiply rows (fan-out). Count `DISTINCT order_id` after joining.
- **Status:** 97% of orders were delivered. Revenue and delivery analysis will use delivered orders.
- **Dates:** orders run from Sept 2016 to Oct 2018, but 2016 and Sept–Oct 2018 are almost
  empty (the data export is incomplete). **Trends will use Jan 2017 – Aug 2018.**
- **Missing delivery dates (2,965)** belong to orders that were never delivered,
  so they are expected, not errors.

In [13]:
con.close()